In [1]:
import pandas as pd
import numpy as np

In [6]:
chemin = r'C:\Users\Deval\Downloads\Radiologue_sein\output\df_raw.csv'
df_raw = pd.read_csv(chemin, sep=';')

display(df_raw.head())


,record_id,departement,code_departement,departement_rubrique,nom_centre,adresse,code_postal,ville,telephone,radiologues,page_source,colonne_source,bloc_source,source_pdf,date_document,statut_extraction,motifs_extraction,texte_bloc_original,bbox,mots_coordonnees
0,p001-b003,Ain,01,Ain,CENTRE D'IMAGERIE MEDICALE - CABINET D'AMBERIE...,RUE ALEXANDRE BÉRARD,1500,AMBERIEU EN BUGEY,04 74 38 50 11,"ACZEL FRANÇOIS, BLINEAU NADINE, CHAMPIN OLIVIE...",1,1,3,C:\Users\Deval\Downloads\Radiologue_sein\PDF F...,2026-09-16,ok,NaN,CENTRE D'IMAGERIE MEDICALE - CABINET\nD'AMBERI...,"[37.897701263427734, 129.46644592285156, 201.4...","[[37.9007568359375, 129.46644592285156, 67.870..."
1,p001-b004,Ain,01,Ain,CH BUGEY SUD - SERVICE DE RADIOLOGIE,700 AVENUE DE NARVICK,1300,BELLEY,04 79 42 59 79,SAYAH ABDELKARIM,1,2,4,C:\Users\Deval\Downloads\Radiologue_sein\PDF F...,2026-09-16,ok,NaN,CH BUGEY SUD - SERVICE DE RADIOLOGIE\n700 AVEN...,"[209.91709899902344, 129.46644592285156, 363.4...","[[209.91709899902344, 129.46644592285156, 220...."
2,p001-b005,Ain,01,Ain,CENTRE DE MAMMOGRAPHIE DE BOURG EN BRESSE,8 AVENUE DU CHAMP DE FOIRE,1000,BOURG EN BRESSE,04 74 22 90 60,"BAAS MUHANAD, BANASTE NATHAN, LOUIS GUILLAUME,...",1,3,5,C:\Users\Deval\Downloads\Radiologue_sein\PDF F...,2026-09-16,ok,NaN,CENTRE DE MAMMOGRAPHIE DE BOURG EN\nBRESSE\n8 ...,"[388.060302734375, 129.46644592285156, 546.838...","[[388.060302734375, 129.46644592285156, 418.02..."
3,p001-b006,Ain,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE L...,15 PLACE ST VINCENT DE PAUL,1400,CHATILLON/CHALARONNE,04 74 38 50 11,"ACZEL FRANÇOIS, BLINEAU NADINE, CHAMPIN OLIVIE...",1,1,6,C:\Users\Deval\Downloads\Radiologue_sein\PDF F...,2026-09-16,ok,NaN,CABINET DE RADIOLOGIE - IMAGERIE\nMÉDICALE DE ...,"[37.897701263427734, 198.7681427001953, 201.48...","[[37.897701263427734, 198.7681427001953, 69.49..."
4,p001-b007,Ain,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE M...,46 RUE DU DR FUVEL,1800,MEXIMIEUX,04 74 38 50 11,"ACZEL FRANÇOIS, BLINEAU NADINE, CHAMPIN OLIVIE...",1,2,7,C:\Users\Deval\Downloads\Radiologue_sein\PDF F...,2026-09-16,ok,NaN,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE\nDE ...,"[209.91709899902344, 198.7681427001953, 382.48...","[[209.91709899902344, 198.7681427001953, 241.5..."


In [10]:
# Code région INSEE : (nom de la région, codes départements)
regions = {
    "01": ("Guadeloupe", "971"),
    "02": ("Martinique", "972"),
    "03": ("Guyane", "973"),
    "04": ("La Réunion", "974"),
    "06": ("Mayotte", "976"),
    "11": ("Île-de-France", "75 77 78 91 92 93 94 95"),
    "24": ("Centre-Val de Loire", "18 28 36 37 41 45"),
    "27": ("Bourgogne-Franche-Comté", "21 25 39 58 70 71 89 90"),
    "28": ("Normandie", "14 27 50 61 76"),
    "32": ("Hauts-de-France", "02 59 60 62 80"),
    "44": ("Grand Est", "08 10 51 52 54 55 57 67 68 88"),
    "52": ("Pays de la Loire", "44 49 53 72 85"),
    "53": ("Bretagne", "22 29 35 56"),
    "75": ("Nouvelle-Aquitaine", "16 17 19 23 24 33 40 47 64 79 86 87"),
    "76": ("Occitanie", "09 11 12 30 31 32 34 46 48 65 66 81 82"),
    "84": ("Auvergne-Rhône-Alpes", "01 03 07 15 26 38 42 43 63 69 73 74"),
    "93": ("Provence-Alpes-Côte d'Azur", "04 05 06 13 83 84"),
    "94": ("Corse", "2A 2B"),
}

# Dictionnaire département → code région
departement_region = {
    dep: code_region
    for code_region, (_, departements) in regions.items()
    for dep in departements.split()
}

df_radiologie = df_raw[
    ["code_departement", "departement", "nom_centre", "ville"]
].copy()

# Normalisation : 1 ou 1.0 → "01", tout en conservant "2A" et "2B"
df_radiologie["code_departement"] = (
    df_radiologie["code_departement"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"\.0+$", "", regex=True)
    .str.zfill(2)
)

df_radiologie["Code_region"] = (
    df_radiologie["code_departement"].map(departement_region).astype("string")
)

df_radiologie["Region"] = df_radiologie["Code_region"].map(
    {code: nom for code, (nom, _) in regions.items()}
)

# Ordre final des colonnes
df_radiologie = df_radiologie[
    ["Code_region", "Region", "code_departement", "departement", "nom_centre", "ville"]
]

display(df_radiologie.head())

# Vérification des codes sans correspondance
sans_region = df_radiologie.loc[
    df_radiologie["Code_region"].isna(), "code_departement"
].drop_duplicates()

if not sans_region.empty:
    print("Codes département sans région :", sans_region.tolist())
df_radiologie.head()

,Code_region,Region,code_departement,departement,nom_centre,ville
0,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE D'IMAGERIE MEDICALE - CABINET D'AMBERIE...,AMBERIEU EN BUGEY
1,84,Auvergne-Rhône-Alpes,01,Ain,CH BUGEY SUD - SERVICE DE RADIOLOGIE,BELLEY
2,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE DE MAMMOGRAPHIE DE BOURG EN BRESSE,BOURG EN BRESSE
3,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE L...,CHATILLON/CHALARONNE
4,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE M...,MEXIMIEUX


Codes département sans région : ['20', 'MC']


,Code_region,Region,code_departement,departement,nom_centre,ville
0,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE D'IMAGERIE MEDICALE - CABINET D'AMBERIE...,AMBERIEU EN BUGEY
1,84,Auvergne-Rhône-Alpes,01,Ain,CH BUGEY SUD - SERVICE DE RADIOLOGIE,BELLEY
2,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE DE MAMMOGRAPHIE DE BOURG EN BRESSE,BOURG EN BRESSE
3,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE L...,CHATILLON/CHALARONNE
4,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE M...,MEXIMIEUX


In [11]:
nb_initial = len(df_radiologie)

# Exclure Monaco
monaco = (
    df_radiologie["code_departement"].astype("string").str.strip().str.upper()
    .isin(["99138", "MC"])
)

for colonne in ["departement", "ville", "nom_centre"]:
    monaco |= (
        df_radiologie[colonne]
        .astype("string")
        .str.contains(r"\bmonaco\b", case=False, na=False)
    )

nb_monaco = int(monaco.sum())
df_radiologie = df_radiologie.loc[~monaco].copy()

# Supprimer les doublons sur l'ensemble des six colonnes
nb_avant = len(df_radiologie)
df_radiologie = df_radiologie.drop_duplicates().reset_index(drop=True)
nb_doublons = nb_avant - len(df_radiologie)

# Traiter les chaînes vides comme des valeurs manquantes
df_radiologie = df_radiologie.replace(r"^\s*$", pd.NA, regex=True)

print(f"Lignes initiales : {nb_initial}")
print(f"Lignes Monaco exclues : {nb_monaco}")
print(f"Doublons supprimés : {nb_doublons}")
print(f"Lignes restantes : {len(df_radiologie)}")

print("\nValeurs manquantes par colonne :")
display(df_radiologie.isna().sum().rename("Nombre de NA").to_frame())

# Afficher les lignes ayant au moins une valeur manquante
lignes_na = df_radiologie.loc[df_radiologie.isna().any(axis=1)]
print(f"\nLignes avec au moins un NA : {len(lignes_na)}")
display(lignes_na)

Lignes initiales : 1657
Lignes Monaco exclues : 1
Doublons supprimés : 85
Lignes restantes : 1571

Valeurs manquantes par colonne :


,Nombre de NA
Code_region,11
Region,11
code_departement,0
departement,0
nom_centre,0
ville,0



Lignes avec au moins un NA : 11


,Code_region,Region,code_departement,departement,nom_centre,ville
220,<NA>,NaN,20,Corse,CENTRE D'IMAGERIE MEDICALE DU CASONE,AJACCIO
221,<NA>,NaN,20,Corse,CENTRE D'IMAGERIE DE LA FEMME - ESPACE MEDICAL...,AJACCIO
222,<NA>,NaN,20,Corse,CABINET DU CENTRE - GROUPEMENT D'IMAGERIE MEDI...,BASTIA
223,<NA>,NaN,20,Corse,SERVICE D'IMAGERIE MÉDICALE - CENTRE HOSPITALI...,BASTIA
224,<NA>,NaN,20,Corse,CABINET D'IMAGERIE MEDICALE BORGO SELARL IMBM,BORGO
225,<NA>,NaN,20,Corse,SERVICE D'IMAGERIE MEDICALE - CENTRE HOSPITALI...,CORTE
226,<NA>,NaN,20,Corse,CABINET DE RADIOLOGIE - IMAGERIE MEDICALE DE L...,ILE ROUSSE
227,<NA>,NaN,20,Corse,CABINET DE RADIOLOGIE LES ALBIZZIAS,PORTO VECCHIO
228,<NA>,NaN,20,Corse,SERVICE D'IMAGERIE MEDICALE POLYCLINIQUE DU SU...,PORTO-VECCHIO
229,<NA>,NaN,20,Corse,CENTRE D'IMAGERIE MEDICALE TIMAVI,PROPRIANO


In [12]:
import json
import re
import unicodedata
from urllib.request import urlopen

def normaliser_ville(valeur):
    if pd.isna(valeur):
        return ""
    texte = unicodedata.normalize("NFKD", str(valeur))
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", texte.lower())

# Référentiel national : permet de détecter les homonymes
url = "https://geo.api.gouv.fr/communes?fields=nom,codeDepartement"
with urlopen(url, timeout=60) as reponse:
    communes = pd.DataFrame(json.load(reponse))

communes["ville_normalisee"] = communes["nom"].map(normaliser_ville)

# Garder uniquement les noms correspondant à une seule commune
communes_uniques = communes.loc[
    ~communes["ville_normalisee"].duplicated(keep=False)
]

correspondance_corse = (
    communes_uniques.loc[
        communes_uniques["codeDepartement"].isin(["2A", "2B"])
    ]
    .set_index("ville_normalisee")["codeDepartement"]
    .to_dict()
)

# Déduire le département à partir de la ville
departement_deduit = (
    df_radiologie["ville"]
    .map(normaliser_ville)
    .map(correspondance_corse)
)

a_corriger = departement_deduit.notna()

df_radiologie.loc[a_corriger, "code_departement"] = (
    departement_deduit.loc[a_corriger]
)

# Compléter la région pour tous les départements corses identifiés
est_corse = df_radiologie["code_departement"].isin(["2A", "2B"])

df_radiologie.loc[est_corse, "departement"] = (
    df_radiologie.loc[est_corse, "code_departement"]
    .map({"2A": "Corse-du-Sud", "2B": "Haute-Corse"})
)
df_radiologie.loc[est_corse, "Code_region"] = "94"
df_radiologie.loc[est_corse, "Region"] = "Corse"

display(
    df_radiologie.loc[
        est_corse,
        ["ville", "code_departement", "departement", "Code_region", "Region"]
    ].drop_duplicates()
)

# Afficher les lignes corses qui restent à vérifier
a_verifier = (
    df_radiologie["code_departement"]
    .astype("string")
    .str.strip()
    .eq("20")
    .fillna(False)
    |
    df_radiologie["departement"]
    .astype("string")
    .str.contains("corse", case=False, na=False)
) & ~est_corse

print(f"Lignes corses restant à vérifier : {a_verifier.sum()}")
display(df_radiologie.loc[a_verifier])

,ville,code_departement,departement,Code_region,Region
220,AJACCIO,2A,Corse-du-Sud,94,Corse
222,BASTIA,2B,Haute-Corse,94,Corse
224,BORGO,2B,Haute-Corse,94,Corse
225,CORTE,2B,Haute-Corse,94,Corse
227,PORTO VECCHIO,2A,Corse-du-Sud,94,Corse
228,PORTO-VECCHIO,2A,Corse-du-Sud,94,Corse
229,PROPRIANO,2A,Corse-du-Sud,94,Corse
230,PRUNELLI DI FIUMORBO,2B,Haute-Corse,94,Corse


Lignes corses restant à vérifier : 1


,Code_region,Region,code_departement,departement,nom_centre,ville
226,<NA>,NaN,20,Corse,CABINET DE RADIOLOGIE - IMAGERIE MEDICALE DE L...,ILE ROUSSE


In [13]:
colonnes_finales = [
    "Code_region",
    "Region",
    "code_departement",
    "departement",
    "nom_centre",
    "ville",
]

df_radiologie_final = (
    df_radiologie[colonnes_finales]
    .replace(r"^\s*$", pd.NA, regex=True)
    .drop_duplicates()
    .sort_values(
        ["Region", "departement", "ville", "nom_centre"],
        na_position="last",
    )
    .reset_index(drop=True)
    .copy()
)

print(f"Nombre de lignes : {len(df_radiologie_final)}")
display(df_radiologie_final)

print("Valeurs manquantes par colonne :")
display(
    df_radiologie_final.isna().sum()
    .rename("Nombre de NA")
    .to_frame()
)

Nombre de lignes : 1571


,Code_region,Region,code_departement,departement,nom_centre,ville
0,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE D'IMAGERIE MEDICALE - CABINET D'AMBERIE...,AMBERIEU EN BUGEY
1,84,Auvergne-Rhône-Alpes,01,Ain,CH BUGEY SUD - SERVICE DE RADIOLOGIE,BELLEY
2,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE DE MAMMOGRAPHIE DE BOURG EN BRESSE,BOURG EN BRESSE
3,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE L...,CHATILLON/CHALARONNE
4,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE M...,MEXIMIEUX
...,...,...,...,...,...,...
1566,11,Île-de-France,78,Yvelines,IMAGERIE MEDICALE VERSAILLES RIVE DROITE SCM,VERSAILLES
1567,11,Île-de-France,78,Yvelines,MAISON MEDICALE CARRE NOTRE DAME,VERSAILLES
1568,11,Île-de-France,78,Yvelines,RESIDENCE GRAND SIECLE,VERSAILLES
1569,11,Île-de-France,78,Yvelines,CENTRE D IMAGERIE MEDICALE VIROFLAY,VIROFLAY


Valeurs manquantes par colonne :


,Nombre de NA
Code_region,1
Region,1
code_departement,0
departement,0
nom_centre,0
ville,0


In [14]:
for df in [df_radiologie, df_radiologie_final]:
    masque = df["ville"].map(normaliser_ville).isin(
        ["ilerousse", "lilerousse"]
    )

    df.loc[masque, [
        "code_departement", "departement", "Code_region", "Region"
    ]] = ["2B", "Haute-Corse", "94", "Corse"]

display(
    df_radiologie_final.loc[
        df_radiologie_final["ville"]
        .map(normaliser_ville)
        .isin(["ilerousse", "lilerousse"])
    ]
)

,Code_region,Region,code_departement,departement,nom_centre,ville
1570,94,Corse,2B,Haute-Corse,CABINET DE RADIOLOGIE - IMAGERIE MEDICALE DE L...,ILE ROUSSE


In [15]:
# Inclure la correction d’Île-Rousse
masque = df_radiologie["ville"].map(normaliser_ville).isin(
    ["ilerousse", "lilerousse"]
)

df_radiologie.loc[masque, [
    "Code_region", "Region", "code_departement", "departement"
]] = ["94", "Corse", "2B", "Haute-Corse"]

# Créer le DataFrame final avec tous les cabinets
df_radiologie_final = (
    df_radiologie[[
        "Code_region",
        "Region",
        "code_departement",
        "departement",
        "nom_centre",
        "ville",
    ]]
    .replace(r"^\s*$", pd.NA, regex=True)
    .drop_duplicates()
    .sort_values(["Region", "departement", "ville", "nom_centre"])
    .reset_index(drop=True)
    .copy()
)

print(f"Nombre de lignes : {len(df_radiologie_final)}")
display(df_radiologie_final)

Nombre de lignes : 1571


,Code_region,Region,code_departement,departement,nom_centre,ville
0,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE D'IMAGERIE MEDICALE - CABINET D'AMBERIE...,AMBERIEU EN BUGEY
1,84,Auvergne-Rhône-Alpes,01,Ain,CH BUGEY SUD - SERVICE DE RADIOLOGIE,BELLEY
2,84,Auvergne-Rhône-Alpes,01,Ain,CENTRE DE MAMMOGRAPHIE DE BOURG EN BRESSE,BOURG EN BRESSE
3,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE L...,CHATILLON/CHALARONNE
4,84,Auvergne-Rhône-Alpes,01,Ain,CABINET DE RADIOLOGIE - IMAGERIE MÉDICALE DE M...,MEXIMIEUX
...,...,...,...,...,...,...
1566,11,Île-de-France,78,Yvelines,HOPITAL DE LA PORTE VERTE,VERSAILLES
1567,11,Île-de-France,78,Yvelines,IMAGERIE MEDICALE VERSAILLES RIVE DROITE SCM,VERSAILLES
1568,11,Île-de-France,78,Yvelines,MAISON MEDICALE CARRE NOTRE DAME,VERSAILLES
1569,11,Île-de-France,78,Yvelines,RESIDENCE GRAND SIECLE,VERSAILLES


In [16]:
import unicodedata

df_radiologie_final = df_radiologie_final.copy()

# 1. Typer les colonnes en texte et nettoyer les valeurs
for col in df_radiologie_final.columns:
    df_radiologie_final[col] = (
        df_radiologie_final[col]
        .astype("string")
        .map(
            lambda x: unicodedata.normalize("NFC", x)
            if pd.notna(x) else pd.NA
        )
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

# 2. Normaliser les codes, sans perdre les zéros initiaux
for col in ["Code_region", "code_departement"]:
    df_radiologie_final[col] = (
        df_radiologie_final[col]
        .str.upper()
        .str.replace(r"\.0+$", "", regex=True)
        .str.zfill(2)
    )

# 3. Harmoniser les libellés géographiques depuis les codes
noms_regions = {code: nom for code, (nom, _) in regions.items()}

df_radiologie_final["Region"] = (
    df_radiologie_final["Code_region"]
    .map(noms_regions)
    .astype("string")
    .fillna(df_radiologie_final["Region"])
)

for code, nom in {"2A": "Corse-du-Sud", "2B": "Haute-Corse"}.items():
    masque = df_radiologie_final["code_departement"].eq(code).fillna(False)
    df_radiologie_final.loc[
        masque, ["departement", "Code_region", "Region"]
    ] = [nom, "94", "Corse"]

# 4. Supprimer les doublons après nettoyage
nb_doublons = int(df_radiologie_final.duplicated().sum())

df_radiologie_final = (
    df_radiologie_final
    .drop_duplicates()
    .sort_values(["Region", "departement", "ville", "nom_centre"])
    .reset_index(drop=True)
)

# 5. Bilan des types et des valeurs manquantes
bilan = pd.DataFrame({
    "Type": df_radiologie_final.dtypes.astype(str),
    "Nombre de NA": df_radiologie_final.isna().sum(),
})

print(f"Lignes finales : {len(df_radiologie_final)}")
print(f"Doublons supprimés : {nb_doublons}")
print(f"Doublons restants : {df_radiologie_final.duplicated().sum()}")
display(bilan)

lignes_na = df_radiologie_final.loc[
    df_radiologie_final.isna().any(axis=1)
]
print(f"Lignes avec au moins un NA : {len(lignes_na)}")
display(lignes_na)

Lignes finales : 1571
Doublons supprimés : 0
Doublons restants : 0


,Type,Nombre de NA
Code_region,string,0
Region,string,0
code_departement,string,0
departement,string,0
nom_centre,string,0
ville,string,0


Lignes avec au moins un NA : 0


,Code_region,Region,code_departement,departement,nom_centre,ville


In [17]:
%pip install -q google-cloud-bigquery pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from google.cloud import bigquery

client = bigquery.Client(project="cancer-du-sein-513216")

table_id = "cancer-du-sein-513216.Data_brute.df_radiologie_final"

schema = [
    bigquery.SchemaField(
        "Code_region", "STRING",
        description="Code officiel INSEE de la région, conservé en texte pour préserver les zéros initiaux."
    ),
    bigquery.SchemaField(
        "Region", "STRING",
        description="Nom de la région administrative où se situe le cabinet de radiologie."
    ),
    bigquery.SchemaField(
        "code_departement", "STRING",
        description="Code officiel INSEE du département : deux caractères en métropole, dont 2A et 2B pour la Corse, ou trois chiffres en outre-mer."
    ),
    bigquery.SchemaField(
        "departement", "STRING",
        description="Nom du département où se situe le cabinet de radiologie."
    ),
    bigquery.SchemaField(
        "nom_centre", "STRING",
        description="Nom du cabinet ou du centre de radiologie, tel que renseigné dans la source."
    ),
    bigquery.SchemaField(
        "ville", "STRING",
        description="Nom de la ville où se situe le cabinet ou le centre de radiologie."
    ),
]

job_config = bigquery.LoadJobConfig(
    schema=schema,
    # Empêche d'écraser ou d'ajouter des données à une table déjà remplie.
    write_disposition=bigquery.WriteDisposition.WRITE_EMPTY,
)

job = client.load_table_from_dataframe(
    df_radiologie_final,
    table_id,
    job_config=job_config,
)
job.result()

# Ajouter la description générale de la table
table = client.get_table(table_id)
table.description = (
    "Répertoire des cabinets et centres de radiologie avec leur localisation "
    "par région, département et ville. Données nettoyées et dédoublonnées "
    "sur les six colonnes ; Monaco exclu."
)
client.update_table(table, ["description"])

table = client.get_table(table_id)
print(f"Table créée : {table.full_table_id}")
print(f"Nombre de lignes dans BigQuery : {table.num_rows}")

Table créée : cancer-du-sein-513216:Data_brute.df_radiologie_final
Nombre de lignes dans BigQuery : 1571
